# 07 Harmonisera och slå ihop

Nu har vi en tvättad tabell per källa. Innan vi kan jämföra dem behöver de tala samma språk. Det kallas att harmonisera. Vi ser till att samma sak har samma nyckel i alla tabeller, att en kolumn betyder samma sak för alla rader och att det syns vilken tidpunkt varje uppgift gäller.

Vi lägger också till en fjärde källa, Kolada, med uppgifter om kommuner och regioner. Till sist slår vi ihop allt till en tabell med en rad per webbplats.

In [ ]:
import json
from pathlib import Path

import httpx
import pandas as pd

RADATA = Path("..") / "data" / "radata"
TVATTAD = Path("..") / "data" / "tvattad"
HARMONISERAD = Path("..") / "data" / "harmoniserad"
HARMONISERAD.mkdir(parents=True, exist_ok=True)

webperf = pd.read_parquet(TVATTAD / "webperf.parquet")
wikidata = pd.read_parquet(TVATTAD / "wikidata.parquet")
crux = pd.read_parquet(TVATTAD / "crux.parquet")

webperf.shape, wikidata.shape, crux.shape

## Kolada

[Kolada](https://www.kolada.se) samlar nyckeltal om Sveriges kommuner och regioner, till exempel från SCB. Varje nyckeltal har ett id. Vi hämtar två.

- `N90002` är Bredbandskollens genomsnittliga hastighet nedströms i Mbit/s, alltså hur snabbt invånarna kan ladda ner. Invånarnas uppkoppling påverkar fältdatan från CrUX men inte labbdatan från Webperf.
- `N01951` är antalet invånare den 31 december. Det använder vi för att kontrollera befolkningen från Wikidata i nästa notebook.

Koladas API finns i version 3. Den äldre version 2 svarar med statuskod 410, Gone, så exempel på nätet som använder `/v2/` fungerar inte längre. Data för ett år publiceras året efter, så det senaste året med data är 2025.

In [ ]:
KOLADA = "https://api.kolada.se/v3"
AR = 2025
NYCKELTAL = {"N90002": "bredband_ned_mbit_s", "N01951": "invanare_kolada"}


def hamta_kolada(nyckeltal, ar):
    svar = httpx.get(f"{KOLADA}/data/kpi/{nyckeltal}/year/{ar}", timeout=60)
    svar.raise_for_status()
    return svar.json()


kolada_svar = {nyckeltal: hamta_kolada(nyckeltal, AR) for nyckeltal in NYCKELTAL}

for nyckeltal, svar in kolada_svar.items():
    print(nyckeltal, svar["count"], len(svar["values"]), svar["next_url"])

`count` är antalet poster och `next_url` adressen till nästa sida om svaret är uppdelat. Här är `next_url` tomt, så vi fick allt på en gång.

Vi sparar rådatan som vanligt.

In [ ]:
hamtad = pd.Timestamp.now(tz="Europe/Stockholm").isoformat(timespec="seconds")

for nyckeltal, svar in kolada_svar.items():
    fil = RADATA / f"kolada-{nyckeltal}-{AR}.json"
    paket = {"hamtad": hamtad, "kalla": KOLADA, "nyckeltal": nyckeltal, "ar": AR, "svar": svar}
    fil.write_text(json.dumps(paket, ensure_ascii=False, indent=2), encoding="utf-8")
    print(fil)

Så här ser en post ut.

In [ ]:
kolada_svar["N01951"]["values"][1]

Fältet `municipality` är Koladas id för kommunen eller regionen. Under `values` finns ett värde per kön. `T` är totalt, `M` män och `K` kvinnor. Vi använder `T`.

Svaret innehåller fler id än kommuner och regioner. Vi räknar hur långa id-numren är.

In [ ]:
id_nummer = pd.Series([post["municipality"] for post in kolada_svar["N01951"]["values"]])

id_nummer.str.len().value_counts()

In [ ]:
id_nummer[id_nummer.str.len() > 4].head()

Id som börjar med `G` är grupper av kommuner som Kolada räknar fram, till exempel alla storstäder. Kommuner och regioner har id med fyra siffror. `0000` är hela riket. Vi behåller id med fyra tecken utom `0000`.

Vi plockar ut värdet för `T` ur varje post och bygger en tabell per nyckeltal, som vi sedan slår ihop.

In [ ]:
def kolada_tabell(svar, kolumn):
    rader = []
    for post in svar["values"]:
        kolada_id = post["municipality"]
        if len(kolada_id) != 4 or kolada_id == "0000":
            continue
        for varde in post["values"]:
            if varde["gender"] == "T":
                rader.append({"kolada_id": kolada_id, kolumn: varde["value"]})
    return pd.DataFrame(rader)


kolada = None
for nyckeltal, kolumn in NYCKELTAL.items():
    tabell = kolada_tabell(kolada_svar[nyckeltal], kolumn)
    kolada = tabell if kolada is None else kolada.merge(tabell, on="kolada_id", how="outer")

kolada["invanare_kolada"] = kolada["invanare_kolada"].astype("Int64")
kolada["kolada_ar"] = AR

kolada.shape, kolada["kolada_id"].is_unique

`None` betyder att variabeln ännu inte har något värde. Första varvet blir `kolada` den första tabellen, och nästa varv slås den andra ihop med den. `how="outer"` behåller id som bara finns i den ena tabellen.

## Samma nyckel för alla

Webperf och Wikidata har samma nyckel, `qid`. Vi börjar med att slå ihop dem.

In [ ]:
tabell = webperf.merge(wikidata, on="qid", how="left", suffixes=("", "_wikidata"))

tabell.shape

Båda tabellerna har en kolumn som heter `hamtad`. `suffixes` bestämmer vad kolumnerna ska heta när namnen krockar. Webperfs behåller sitt namn och Wikidatas får tillägget `_wikidata`.

Kolada använder andra nycklar. För en kommun är Koladas id kommunkoden, till exempel `0180` för Stockholm. För en region är det `00` följt av länskoden, till exempel `0001` för Region Stockholm. Vi bygger en kolumn `kolada_id` efter den regeln.

`where` behåller värdet där villkoret är sant och tar annars värdet från det andra argumentet. Här tar vi kommunkoden för kommuner och `00` plus länskoden för regioner.

In [ ]:
tabell["kolada_id"] = tabell["kommunkod"].where(tabell["typ"] == "kommun", "00" + tabell["lanskod"])

tabell[["namn", "typ", "kommunkod", "lanskod", "kolada_id"]].iloc[[0, 1, 289, 290]]

`iloc` väljer rader efter deras plats i tabellen. Här visar vi två kommuner och två regioner.

## En kolumn betyder samma sak

Vi vill ha en kolumn med befolkning för alla rader. För kommunerna är det kommunens befolkning och för regionerna länets. Vi använder `where` på samma sätt och sparar i en egen kolumn vad befolkningen avser, så att det inte går att missa.

In [ ]:
ar_kommun = tabell["typ"] == "kommun"

tabell["befolkning"] = tabell["befolkning"].where(ar_kommun, tabell["lan_befolkning"])
tabell["befolkning_datum"] = tabell["befolkning_datum"].where(ar_kommun, tabell["lan_befolkning_datum"])
tabell["befolkning_avser"] = "kommunen"
tabell.loc[~ar_kommun, "befolkning_avser"] = "länet"

tabell = tabell.drop(columns=["lan_befolkning", "lan_befolkning_datum"])

tabell[["namn", "typ", "befolkning", "befolkning_datum", "befolkning_avser"]].iloc[[0, 289, 290]]

`~` betyder inte, så `~ar_kommun` är sant för regionerna. Med `loc` sätter vi ett värde på de rader där villkoret är sant.

Gotland är både kommun och region. Webperf räknar Gotland som region, så Gotland får länets befolkning. Kommunen och länet omfattar samma område, så det blir samma tal.

In [ ]:
tabell[tabell["namn"].str.contains("Gotland")][["namn", "typ", "kommunkod", "kolada_id", "befolkning", "befolkning_avser"]]

## Slå ihop med CrUX och Kolada

Nu kopplar vi på CrUX via `origin` och Kolada via `kolada_id`. Med `indicator=True` får vi en kolumn `_merge` som berättar om raden hittades i båda tabellerna, `both`, eller bara i vår tabell, `left_only`.

In [ ]:
tabell = tabell.merge(crux, on="origin", how="left", suffixes=("", "_crux"), indicator=True)
print(tabell["_merge"].value_counts())
tabell = tabell.drop(columns="_merge")

tabell = tabell.merge(kolada, on="kolada_id", how="left", indicator=True)
print(tabell["_merge"].value_counts())
tabell = tabell.drop(columns="_merge")

tabell.shape

Alla 310 webbplatser hittades i både CrUX och Kolada. Att två webbplatser saknar data i CrUX syns i stället i kolumnen `crux_status`, eftersom vi behöll dem när vi tvättade.

## Vilken tid gäller uppgifterna?

Källorna mäter vid olika tidpunkter. Det behöver vi ha med oss när vi jämför. Vi samlar tidpunkterna i en liten tabell.

In [ ]:
tider = pd.DataFrame([
    {"kalla": "Webperf", "avser": "betyg vid hämtningen", "tid": tabell["hamtad"].min().date()},
    {"kalla": "Wikidata", "avser": "befolkning", "tid": tabell["befolkning_datum"].max().date()},
    {"kalla": "CrUX", "avser": "28 dagar fram till", "tid": tabell["period_slut"].max().date()},
    {"kalla": "Kolada", "avser": "invånare 31 december och bredband under året", "tid": AR},
])

tider

Webperfs betyg och CrUX gäller ungefär samma tid. Befolkningen från Wikidata och Kolada gäller olika dagar, med ett halvår emellan. Bredbandsmätningarna gäller hela 2025.

## Den sammanslagna tabellen

Vi väljer ut kolumnerna i en ordning som är lätt att läsa, med en rad per webbplats.

In [ ]:
kolumner = [
    "webperf_id", "namn", "typ", "webbplats", "origin",
    "qid", "kommunkod", "kolada_id", "lan", "lanskod",
    "befolkning", "befolkning_datum", "befolkning_avser", "invanare_kolada",
    "betyg_totalt", "betyg_tillganglighet", "betyg_prestanda", "betyg_sakerhet", "betyg_standard",
    "crux_status", "lcp_p75_ms", "inp_p75_ms", "cls_p75", "period_start", "period_slut",
    "bredband_ned_mbit_s", "kolada_ar",
    "hamtad", "hamtad_wikidata", "hamtad_crux",
]

webbplatser = tabell[kolumner]

webbplatser.shape, webbplatser["webperf_id"].is_unique

In [ ]:
webbplatser.head()

## Spara

Vi sparar den sammanslagna tabellen som Parquet i mappen `data/harmoniserad`.

In [ ]:
fil = HARMONISERAD / "webbplatser.parquet"
webbplatser.to_parquet(fil, index=False)
fil

## Det här vet vi nu

- Att harmonisera betyder att källorna får samma nycklar, att en kolumn betyder samma sak för alla rader och att det syns vilken tid uppgifterna gäller.
- Kolada har kommunkoden som id för kommuner och `00` plus länskoden för regioner.
- Regionerna får länets befolkning, och det står i en egen kolumn.
- `indicator=True` visar om en sammanslagning hittade alla rader.
- Källorna mäter vid olika tidpunkter.

I nästa notebook validerar vi den sammanslagna tabellen, bland annat genom att jämföra befolkningen från Wikidata med Kolada.